# Jembatan Pemahaman: Dari CIFAR-10 ke Protein Q3

Notebook ini dibuat untuk menghubungkan pemahaman Anda dari [cnn.ipynb](C:\Users\Zainal\Desktop\protein-cnn\cnn.ipynb) ke tugas prediksi struktur sekunder protein pada [Protein_1D_Q3_InProperWay.ipynb](C:\Users\Zainal\Desktop\protein-cnn\exp_beginner\Protein_1D_Q3_InProperWay.ipynb).

## Ide utama
- Pada CIFAR-10, model belajar dari **gambar 2D** untuk memprediksi **1 label per gambar**.
- Pada protein Q3, model belajar dari **urutan 1D residu** untuk memprediksi **1 label per residu**.
- CNN-nya tetap sama idenya: mencari **pola lokal**.
- Bedanya ada pada **bentuk data**, **arti fitur**, dan **target prediksi**.

## Peta konsep cepat
- CIFAR-10: `gambar 32x32x3` -> `Conv2D` -> `Dense` -> `10 kelas`
- Protein Q3: `urutan 700x54` -> `Conv1D` -> `TimeDistributed(Dense)` -> `3 kelas per posisi`

## Analogi sederhana
- CIFAR: “Gambar ini pesawat, kucing, atau truk?”
- Protein: “Residu ke-1 helix/sheet/coil? Residu ke-2 apa? Residu ke-3 apa?”


# Protein Secondary Structure Prediction - Q3 (Proper Research Methodology)

**Task**: Q3 classification (Helix, Sheet, Coil) with proper train/val/test split

**Improvements from original**:
- Proper data split: CullPDB → Train/Val, CB513 = Test holdout (no leakage)
- Best model checkpointing (not last epoch)
- Early stopping with patience
- Fixed seeds for reproducibility
- True test evaluation (CB513 evaluated once at end)
- Model saving and results tracking

In [ ]:
# Setup untuk Vast.ai / Local GPU Environment
import os

# Path ke dataset (sesuaikan dengan lokasi data Anda)
base_dir = "/workspace/data"  # Ganti sesuai path data Anda

# Buat output directory
output_dir = "/workspace/outputs"
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Environment setup complete")
print(f"   Dataset path: {base_dir}")
print(f"   Output path: {output_dir}")

In [ ]:
# Install dependencies (run once)
!pip install tensorflow numpy matplotlib -q

import tensorflow as tf
print(f"✅ TensorFlow {tf.__version__} installed")
print(f"🎮 GPU Available: {tf.config.list_physical_devices('GPU')}")

## Menghubungkan Bagian Load Data dengan CIFAR-10

Pada CIFAR-10, Anda terbiasa dengan:

```python
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
```

Artinya:
- `train_images` dan `test_images` adalah input
- `train_labels` dan `test_labels` adalah target

Pada notebook protein, bentuknya berbeda:

```python
import numpy as np
import random
import json
from datetime import datetime

base_dir = "/content/drive/MyDrive/Raw Dataset"

cb513 = np.load(base_dir + "/cb513+profile_split1.npy.gz", allow_pickle=True)
cullpdb = np.load(base_dir + "/cullpdb+profile_5926_filtered.npy.gz", allow_pickle=True)

print(f"CB513 shape: {cb513.shape}")
print(f"CullPDB shape: {cullpdb.shape}")
```

## Tujuan
Memuat dataset protein mentah dari file `.npy.gz`.

## Cara membacanya
- `base_dir` adalah folder tempat dataset berada.
- `np.load(...)` membaca file NumPy yang sudah disimpan sebelumnya.
- `cb513` dan `cullpdb` adalah dua kumpulan data protein.

## Analogi dengan CIFAR-10
- `cullpdb` kira-kira seperti sumber utama data latih.
- `cb513` dipakai sebagai test holdout, mirip `test_images/test_labels`, tetapi disimpan dalam file terpisah.

## Perbedaan penting
Pada CIFAR, input dan label sudah dipisahkan saat load data. Pada dataset protein ini, input dan label masih menyatu dalam satu tensor mentah, sehingga nanti harus dipisahkan manual.


In [ ]:
import numpy as np
import random
import json
from datetime import datetime

# Load dataset dari local path
cb513 = np.load(base_dir + "/cb513+profile_split1.npy.gz", allow_pickle=True)
cullpdb = np.load(base_dir + "/cullpdb+profile_5926_filtered.npy.gz", allow_pickle=True)

print(f"CB513 shape: {cb513.shape}")
print(f"CullPDB shape: {cullpdb.shape}")

## Kenapa Harus `reshape`?

Setelah data dimuat, notebook melakukan:

```python
cb513 = cb513.reshape(cb513.shape[0], 700, 57)
cullpdb = cullpdb.reshape(cullpdb.shape[0], 700, 57)
```

## Tujuan
Mengubah data mentah menjadi format 3D yang bisa dibaca model.

## Cara membacanya
- dimensi 1: jumlah sampel protein
- dimensi 2: panjang maksimum sekuens = `700` residu
- dimensi 3: `57` fitur per residu

## Analogi dengan CIFAR-10
- CIFAR: `(jumlah_gambar, 32, 32, 3)`
- Protein: `(jumlah_protein, 700, 57)`

Bedanya:
- CIFAR punya struktur spasial 2D: tinggi x lebar
- Protein punya struktur urutan 1D: posisi residu dari kiri ke kanan


In [ ]:
# Reshape to (samples, 700, 57)
cb513 = cb513.reshape(cb513.shape[0], 700, 57)
cullpdb = cullpdb.reshape(cullpdb.shape[0], 700, 57)

print(f"CB513 reshaped: {cb513.shape}")
print(f"CullPDB reshaped: {cullpdb.shape}")

## Split Data: Padanan Train/Validation/Test pada CIFAR

Di CIFAR notebook, pembagian train dan test sudah tersedia dari dataset.

Di notebook protein ini, split dibuat manual:
- `CullPDB` dibagi menjadi `train` dan `validation`
- `CB513` disimpan sebagai `test holdout`

## Mengapa ini penting?
Karena test set harus benar-benar tidak ikut dipakai selama training. Kalau test dipakai saat validasi, hasil bisa bias dan terlalu optimistis.

## Padanan konsep
- CIFAR `train_images` -> `dataset_train`
- CIFAR validation tambahan -> `dataset_val`
- CIFAR `test_images` -> `dataset_test = cb513`


In [ ]:
# ==========================================
# PROPER DATA SPLIT (No Data Leakage)
# ==========================================

# FIXED SEED for reproducible split
SPLIT_SEED = 42
np.random.seed(SPLIT_SEED)

# Split CullPDB: 80% train, 20% validation
n_samples = cullpdb.shape[0]
indices = np.random.permutation(n_samples)
split_idx = int(0.8 * n_samples)

train_idx = indices[:split_idx]
val_idx = indices[split_idx:]

print(f"Total CullPDB samples: {n_samples}")
print(f"Train indices: {len(train_idx)} ({len(train_idx)/n_samples*100:.1f}%)")
print(f"Val indices: {len(val_idx)} ({len(val_idx)/n_samples*100:.1f}%)")
print(f"\nCB513 (TEST - HOLDOUT): {cb513.shape[0]} samples (NEVER seen during training)\n")

# Split the data
dataset_train = cullpdb[train_idx]
dataset_val = cullpdb[val_idx]
dataset_test = cb513  # Reserved for final test only

print(f"Dataset shapes:")
print(f"  Train: {dataset_train.shape}")
print(f"  Val: {dataset_val.shape}")
print(f"  Test (CB513): {dataset_test.shape}")

## Bagaimana Membaca `extract_features_and_labels()`

Ini bagian yang biasanya membuat notebook protein terasa lebih asing daripada CIFAR.

Pada CIFAR:
- `X` = gambar
- `y` = satu label per gambar

Pada protein:
- `X` = fitur untuk setiap residu dalam satu protein
- `y` = label Q3 untuk setiap residu
- `mask` = penanda posisi valid, karena tidak semua protein panjangnya 700

## Makna biologinya
- `Q3` berarti 3 kelas struktur sekunder:
  - `Helix`
  - `Sheet`
  - `Coil`

## Kenapa ada mask?
Protein asli panjangnya berbeda-beda. Agar bisa diproses batch, semuanya dipadding sampai panjang 700. `mask` dipakai agar bagian padding tidak ikut dihitung sebagai data nyata.

## Analogi dengan CIFAR
Kalau CIFAR semua gambar dipaksa ukuran sama `32x32`, protein juga dipaksa panjang sama `700`, tetapi di ujung sekuens ada bagian kosong. Itulah yang harus diabaikan dengan `mask`.


In [ ]:
# ==========================================
# FEATURE EXTRACTION
# ==========================================

def extract_features_and_labels(dataset):
    """
    Extract features and Q3 labels from dataset
    Q3 labels: indices 21:24 (Helix, Sheet, Coil)
    Features: 21 amino acids + 33 other features = 54 total
    """
    # Q3 labels (3 classes)
    y = dataset[:, :, 21:24]
    
    # Features: 0-20 (amino acids) + 24-56 (other features)
    X_aa = dataset[:, :, :21]  # amino acid one-hot
    X_other = dataset[:, :, 24:]  # other features (33 dims)
    X = np.concatenate([X_aa, X_other], axis=2)
    
    # Mask for valid residues (non-padding)
    mask = np.sum(X, axis=2) != 0
    
    return X.astype(np.float32), y.astype(np.float32), mask

# Extract for all splits
X_train, y_train, mask_train = extract_features_and_labels(dataset_train)
X_val, y_val, mask_val = extract_features_and_labels(dataset_val)
X_test, y_test, mask_test = extract_features_and_labels(dataset_test)

print("Feature extraction complete:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"\nMask coverage:")
print(f"  Train: {mask_train.sum() / mask_train.size * 100:.1f}% valid residues")
print(f"  Val: {mask_val.sum() / mask_val.size * 100:.1f}% valid residues")
print(f"  Test: {mask_test.sum() / mask_test.size * 100:.1f}% valid residues")

In [ ]:
# ==========================================
# REPRODUCIBILITY: FIXED SEEDS
# ==========================================

import tensorflow as tf

# Fixed seeds for reproducibility
GLOBAL_SEED = 42

# Python random
random.seed(GLOBAL_SEED)

# NumPy
np.random.seed(GLOBAL_SEED)

# TensorFlow
tf.random.set_seed(GLOBAL_SEED)

print(f"✅ Seeds fixed to {GLOBAL_SEED} for reproducibility")
print(f"   - Python random: seeded")
print(f"   - NumPy: seeded")
print(f"   - TensorFlow: seeded")

## Dari `Conv2D` ke `Conv1D`

Pada CIFAR, Anda melihat `Conv2D` karena model menyapu filter di atas gambar 2D.

Pada protein, notebook memakai:

```python
x = Conv1D(64, 3, padding='same', activation='relu')(input_layer)
x = Conv1D(128, 3, padding='same', activation='relu')(x)
```

## Tujuan
Mencari pola lokal antar residu yang berdekatan.

## Analogi dengan CIFAR
- `Conv2D`: melihat pola lokal antar piksel bertetangga
- `Conv1D`: melihat pola lokal antar residu bertetangga

Kernel size `3` bisa dibayangkan sebagai model yang membaca konteks kecil, misalnya residu kiri, tengah, dan kanan pada satu posisi.

## Kenapa output-nya `TimeDistributed`?
Karena targetnya bukan satu label untuk satu protein, melainkan satu label untuk setiap posisi residu. Jadi prediksi dilakukan di setiap langkah urutan.


In [ ]:
# ==========================================
# MODEL ARCHITECTURE (Same as original, but Q3 output)
# ==========================================

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, Dropout, TimeDistributed, Dense

input_layer = Input(shape=(700, 54))

x = Conv1D(64, 3, padding='same', activation='relu')(input_layer)
x = Conv1D(128, 3, padding='same', activation='relu')(x)

x = Dropout(0.3)(x)

x = TimeDistributed(Dense(64, activation='relu'))(x)
output = TimeDistributed(Dense(3, activation='softmax'))(x)

model = Model(inputs=input_layer, outputs=output)

model.summary()

In [ ]:
# ==========================================
# COMPILE MODEL
# ==========================================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled successfully")

In [ ]:
# ==========================================
# CALLBACKS: CHECKPOINTING + EARLY STOPPING
# ==========================================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. ModelCheckpoint: Save best model based on validation accuracy
checkpoint_callback = ModelCheckpoint(
    filepath=f'{output_dir}/best_model_q3_proper.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1,
    save_weights_only=False
)

# 2. EarlyStopping: Stop if no improvement
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

callbacks = [checkpoint_callback, early_stop_callback]

print("✅ Callbacks configured:")
print(f"   - ModelCheckpoint: saves best val_accuracy model")
print(f"   - EarlyStopping: patience=5, restore_best_weights=True")

## Training: Mirip CIFAR, tetapi Ada `sample_weight`

Struktur training tetap familiar:
- compile model
- fit model
- lihat `history`
- evaluasi model

Perbedaan penting:

```python
sample_weight_train = mask_train.astype(np.float32)
sample_weight_val = mask_val.astype(np.float32)
```

## Tujuan
Memastikan padding tidak ikut memengaruhi loss dan accuracy.

## Cara memahaminya
Di CIFAR, setiap piksel pada gambar adalah data nyata. Pada protein, tidak semua posisi sampai indeks 700 adalah residu sungguhan. Jadi model harus “dibutakan” terhadap padding.


In [ ]:
# ==========================================
# TRAINING (with validation, NOT on CB513)
# ==========================================

print("\n" + "="*60)
print("TRAINING START")
print("="*60)
print(f"Train samples: {X_train.shape[0]}")
print(f"Val samples: {X_val.shape[0]}")
print(f"Test samples (CB513 - HOLDOUT): {X_test.shape[0]}")
print("\n⚠️  CB513 is RESERVED for final test only!\n")

# Prepare sample weights (for masking padding)
sample_weight_train = mask_train.astype(np.float32)
sample_weight_val = mask_val.astype(np.float32)

# Train with callbacks
history = model.fit(
    X_train, y_train,
    sample_weight=sample_weight_train,
    validation_data=(X_val, y_val, sample_weight_val),
    epochs=50,  # High max, early stopping will handle it
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

In [ ]:
# ==========================================
# LOAD BEST MODEL (not last epoch)
# ==========================================

from tensorflow.keras.models import load_model

# Load the best model (saved by ModelCheckpoint)
best_model_path = f'{output_dir}/best_model_q3_proper.h5'
model = load_model(best_model_path)

print(f"✅ Loaded best model from: {best_model_path}")
print(f"\nTraining history summary:")
print(f"   Total epochs: {len(history.history['loss'])}")
print(f"   Best val accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"   Best val epoch: {np.argmax(history.history['val_accuracy']) + 1}")

## Evaluasi: Jangan Terkecoh Akurasi Training

Pelajaran dari CIFAR tetap berlaku:
- `train accuracy` tinggi belum tentu model bagus
- yang penting adalah `validation accuracy` dan `test accuracy`

Pada notebook ini:
- `validation` dipakai untuk memilih model terbaik
- `CB513` dipakai sekali di akhir sebagai test holdout

## Kenapa metodologi ini bagus?
Karena hasil test lebih jujur dan lebih layak untuk laporan atau skripsi.


In [ ]:
# ==========================================
# FINAL TEST EVALUATION (CB513 - EVALUATED ONCE)
# ==========================================

print("\n" + "="*60)
print("🧪 FINAL TEST EVALUATION")
print("="*60)
print("Dataset: CB513 (HOLDOUT - never seen during training)")
print("Evaluation: ONE TIME ONLY (proper methodology)\n")

# Evaluate on test set
sample_weight_test = mask_test.astype(np.float32)
test_results = model.evaluate(
    X_test, y_test,
    sample_weight=sample_weight_test,
    verbose=1
)

test_loss = test_results[0]
test_accuracy = test_results[1]

print("\n" + "="*60)
print("📊 FINAL TEST RESULTS")
print("="*60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy (Q3): {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("\n✅ Proper evaluation complete!")
print("   (CB513 was never used during training/validation)")

In [ ]:
# ==========================================
# SAVE RESULTS & REPORT
# ==========================================

# Create comprehensive report
report = {
    "experiment_info": {
        "name": "Q3_Protein_Secondary_Structure_Proper_Methodology",
        "date": datetime.now().isoformat(),
        "seed": GLOBAL_SEED,
        "task": "Q3_classification",
        "architecture": "CNN1D_Q3"
    },
    "data_split": {
        "train": X_train.shape[0],
        "val": X_val.shape[0],
        "test": X_test.shape[0],
        "split_method": "CullPDB_80_20_CB513_holdout",
        "split_seed": SPLIT_SEED
    },
    "training_config": {
        "epochs_max": 50,
        "epochs_actual": len(history.history['loss']),
        "early_stopping_patience": 5,
        "batch_size": 16,
        "optimizer": "adam",
        "loss": "categorical_crossentropy"
    },
    "results": {
        "best_val_accuracy": float(max(history.history['val_accuracy'])),
        "best_val_epoch": int(np.argmax(history.history['val_accuracy']) + 1),
        "best_val_loss": float(min(history.history['val_loss'])),
        "final_train_accuracy": float(history.history['accuracy'][-1]),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "test_accuracy_percent": float(test_accuracy * 100)
    },
    "training_history": {
        "loss": [float(x) for x in history.history['loss']],
        "accuracy": [float(x) for x in history.history['accuracy']],
        "val_loss": [float(x) for x in history.history['val_loss']],
        "val_accuracy": [float(x) for x in history.history['val_accuracy']]
    },
    "improvements_over_original": [
        "Proper train/val/test split (no CB513 leakage)",
        "Best model checkpointing (not last epoch)",
        "Early stopping with patience=5",
        "Fixed seeds for reproducibility",
        "True test evaluation (CB513 once at end)",
        "Model saving and comprehensive reporting"
    ]
}

# Save to output directory
report_path = f'{output_dir}/q3_proper_results.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✅ Report saved to: {report_path}")
print("\n📋 Report Summary:")
print(json.dumps(report['results'], indent=2))

In [ ]:
# ==========================================
# VISUALIZATION: TRAINING CURVES
# ==========================================

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
ax1.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax1.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
ax1.axvline(x=np.argmin(history.history['val_loss']), color='r', linestyle='--', 
            label=f'Best Val Loss (Epoch {np.argmin(history.history["val_loss"]) + 1})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(history.history['accuracy'], label='Train Acc', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
ax2.axvline(x=np.argmax(history.history['val_accuracy']), color='r', linestyle='--',
            label=f'Best Val Acc (Epoch {np.argmax(history.history["val_accuracy"]) + 1})')
ax2.axhline(y=test_accuracy, color='g', linestyle=':',
            label=f'Test Acc: {test_accuracy:.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy (Q3)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save plot
plot_path = f'{output_dir}/q3_training_curves.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"\n✅ Plot saved to: {plot_path}")

In [ ]:
# ==========================================
# COMPARISON WITH ORIGINAL (Summary)
# ==========================================

print("\n" + "="*70)
print("📊 COMPARISON: Original vs Proper Methodology")
print("="*70)

comparison = f"""
METRIC                          | ORIGINAL (WRONG)    | PROPER WAY (THIS)
--------------------------------------------------------------------------------
Data Split:                     | CB513 = Validation  | CullPDB→Train/Val, CB513=Test
Data Leakage:                   | ✅ YES (80%+ seen)  | ❌ NO (truly holdout)
Model Selection:                | Last epoch          | Best val checkpoint
Early Stopping:                | ❌ No (fixed 20 ep) | ✅ Yes (patience=5)
Reproducibility:                | ❌ No seed         | ✅ Seed=42
Test Evaluation:                | ❌ On CB513 (val)   | ✅ Once at end
Model Saved:                    | ❌ No              | ✅ Yes (.h5)
Report:                         | ❌ No              | ✅ Full JSON
Training Curves:                | ❌ No              | ✅ Saved plot
--------------------------------------------------------------------------------
THIS EXPERIMENT:                                   | RESULTS
--------------------------------------------------------------------------------
Best Validation Accuracy:                           | {max(history.history['val_accuracy']):.4f}
Final Test Accuracy (CB513, true holdout):          | {test_accuracy:.4f} ({test_accuracy*100:.2f}%)
Training Epochs (with early stopping):              | {len(history.history['loss'])}
--------------------------------------------------------------------------------
"""

print(comparison)

print("\n✨ Key Takeaway:")
print(f"   Original '81%' was on CB513 used as VALIDATION (data leakage)")
print(f"   Proper test accuracy: {test_accuracy*100:.2f}% (true generalization)")
print(f"   Difference: ~{81 - test_accuracy*100:.1f} percentage points from data leakage!")